## Task 1: Load and Prepare Text

In [10]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF document
loader = PyPDFLoader("cover.pdf")
documents = loader.load()

# Combine text from all pages
text = "\n".join(doc.page_content for doc in documents)

# Print total characters
print("Total Characters:", len(text))

# Print sample content preview
print("\nSample Content Preview:")
print(text[:500])

Total Characters: 1777

Sample Content Preview:
Plain Concepts’ commitment to building custom, technically ambitious solutions — not 
just outsourcing — stands out in a space where many settle for off-the-shelf approaches.
Over the past five years, I’ve led and contributed to end-to-end Flutter development 
across AI-driven and scalable mobile applications. At Finva Tech, I currently lead a team 
of Flutter developers on an AI-integrated mobile platform, guiding architecture deci-
sions, ensuring timely delivery, and embedding features that e


## Task 2: Prompt-Based Summarization

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# Initialize LLM
llm = ChatGroq(model="openai/gpt-oss-120b")

prompt = PromptTemplate(
    template='''
You are an expert text summarizer.
Summarize the following text clearly and concisely.
Include only the main ideas.
Text: 
{text}
''',
input_variables=['text']
)

# Create the chain
chain = prompt | llm

# Generate summary
response = chain.invoke({"text": text})

print(response.content)

**Summary**

- Plain Concepts distinguishes itself by delivering custom, technically ambitious solutions rather than off‑the‑shelf products.  
- Over the last five years, the author has led end‑to‑end Flutter development for AI‑driven, scalable mobile apps.  
- Currently at Finva Tech, they lead a Flutter team, make architectural decisions, ensure timely delivery, and enhance user experience with AI integration.  
- Previously at Act T Connect, they optimized performance, mentored junior developers, and promoted new Flutter capabilities for scalability and maintainability.  
- Since 2018, they have solid expertise in Flutter, Dart, Firebase, and REST APIs, covering the full development lifecycle and applying best‑practice state management.  
- They have strong cross‑team collaboration and mentorship experience, aligning with Plain Concepts’ focus on technical excellence and knowledge sharing.  
- Although new to a client‑facing consultancy role, they have led cross‑functional teams, tr

## Task 3: Prompt Variations

In [13]:
short_prompt = PromptTemplate(
    template="""
Summarize the following text in 5-6 lines.
Include only the most important information.
Text:
{text}
""",
input_variables=["text"],
)

short_chain = short_prompt | llm

short_response = short_chain.invoke({'text': text})
print("SHORT SUMMARY:")
print(short_response.content)

bullet_prompt = PromptTemplate(
    template="""
Summarize the following text using 5-7 bullet points.
Each bullet should contain one key idea.
Text:
{text}
""",
input_variables=["text"],
)

bullet_chain = bullet_prompt | llm

bullet_response = bullet_chain.invoke({'text': text})
print("BULLET SUMMARY:")
print(bullet_response.content)


SHORT SUMMARY:
- Manish Agrahari highlights his five‑year track record delivering end‑to‑end Flutter apps, especially AI‑driven, scalable mobile solutions.  
- Currently leads a Flutter team at Finva Tech, shaping architecture, ensuring on‑time delivery, and enhancing user experience on an AI‑integrated platform.  
- Previously at Act T Connect, he improved performance, mentored junior developers, and introduced new Flutter features for better scalability and maintainability.  
- Proficient in Flutter, Dart, Firebase, REST APIs, and full‑cycle development—from architecture and UI/UX design to integration and release—using clean code and solid state‑management practices.  
- Experienced in cross‑functional collaboration and knowledge‑sharing; although new to a client‑facing consultancy model, he is confident in adapting to Plain Concepts’ agile, multidisciplinary environment and contributing immediately.
BULLET SUMMARY:
- Plain Concepts differentiates itself by delivering custom, techni

## Task 4: Why Stuff Chain Is Needed

Stuff Chain: A Stuff Chain combines all input documents into a single prompt and sends them to the LLM in one call to generate a summary.


Suitable to use:
1. It provides a simple way to summarize documents.
2. It preserves the overall context across the provided documents.
3. It is suitable for short documents that fit within the model's context window.

Limitation:
1. Large documents may exceed the LLM's context window.
2. Sending a large amount of text in one prompt can increase token usage and cost.


## Task 5: Implement Stuff Summarization Chain

In [15]:
from langchain_classic.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm=llm, chain_type='stuff', verbose=True, prompt=prompt)

stuff_response = chain.run(documents)

print(stuff_response)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert text summarizer.
Summarize the following text clearly and concisely.
Include only the main ideas.
Text: 
Plain Concepts’ commitment to building custom, technically ambitious solutions — not 
just outsourcing — stands out in a space where many settle for off-the-shelf approaches.
Over the past five years, I’ve led and contributed to end-to-end Flutter development 
across AI-driven and scalable mobile applications. At Finva Tech, I currently lead a team 
of Flutter developers on an AI-integrated mobile platform, guiding architecture deci-
sions, ensuring timely delivery, and embedding features that elevate user experience. 
Prior to that, at Act T Connect, I played a key role in optimizing performance and 
mentoring junior developers while driving the adoption of new Flutter capabilities to 
improve scalability and maintainability.
My foundation in mobile developme

## Task 6: Comparison with Prompt-Based Summary

In [16]:

# Task 6: Comparison with Prompt-Based Summary

# Summary from Task 2
prompt_summary = response.content

# Summary from Task 5
stuff_summary = stuff_response

print("1. PROMPT-BASED SUMMARY:\n")
print(prompt_summary)

print("\n2. STUFF CHAIN SUMMARY:\n")
print(stuff_summary)

# The Stuff Chain handles LangChain Document objects and combines their content automatically, while the basic prompt approach requires you to pass the text explicitly.

1. PROMPT-BASED SUMMARY:

**Summary**

- Plain Concepts distinguishes itself by delivering custom, technically ambitious solutions rather than off‑the‑shelf products.  
- Over the last five years, the author has led end‑to‑end Flutter development for AI‑driven, scalable mobile apps.  
- Currently at Finva Tech, they lead a Flutter team, make architectural decisions, ensure timely delivery, and enhance user experience with AI integration.  
- Previously at Act T Connect, they optimized performance, mentored junior developers, and promoted new Flutter capabilities for scalability and maintainability.  
- Since 2018, they have solid expertise in Flutter, Dart, Firebase, and REST APIs, covering the full development lifecycle and applying best‑practice state management.  
- They have strong cross‑team collaboration and mentorship experience, aligning with Plain Concepts’ focus on technical excellence and knowledge sharing.  
- Although new to a client‑facing consultancy role, they have led 

## Task 7: Why Map-Reduce Is Needed

Map-Reduce is a document summarization technique that divides a large document into smaller chunks, summarizes each chunk separately, and then combines those summaries to generate a final summary.

Large document needs Map_reduce because it can process large documents that may not fit into a single LLM prompt.

It works :
1. Divides a large document into smaller chunks.
2. Summarizes each chunk separately.
3. Combines those summaries to generate a final summary.

## Task 8: Implement Map-Reduce Summarization Chain

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

split_docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(split_docs))

# Create Map-Reduce summarization chain
map_reduce_chain = load_summarize_chain(llm=llm, verbose=True, chain_type='map_reduce', map_prompt=prompt, combine_prompt=prompt)

# Generate final summary
map_reduce_response = map_reduce_chain.run(split_docs)

print(map_reduce_response)

## Task 10: Understanding Refine Chain

1. Refine summarization generates an initial summary from the first document chunk. It then processes each chunk and updates the existing summary with any new, relevant information. This process continues until all chunks have been processed.


2. Difference between Map-Reduce and Refine

Map-Reduce:
Summarizes chunks independently, then combines them.

Chunk summaries are generated separately

Large documents with independent sections

May lose connections between sections


Refine:
Updates the summary sequentially.

Each refinement uses the current summary and new chunk

Documents where information builds progressively

later chunks may influence the summary more than earlier ones

## Task 11: Implement Refine Summarization Chain

In [ ]:
initial_prompt = PromptTemplate(
    template='''
Write a concise summary of the following text: \n {text}
''',
input_variables=['text']
)

refine_prompt = PromptTemplate(
    template='''
Your current summary is {existing_answer}
Now refine and improve it using the following new text: {text}
Provide an updated improved summary that include important details from old and new text
''',
input_variables=['existing_answer', 'text']
)

refine_chain = load_summarize_chain(llm=llm, chain_type='refine', verbose=True, question_prompt=initial_prompt, refine_prompt=refine_prompt)

refine_response = refine_chain.run(split_docs)

print(refine_response)


## Task 12: Comparison of All Summarization Methods

In [ ]:

# Collect summaries from previous tasks
summaries = {
    "Prompt-Based": response.content,
    "Stuff Chain": stuff_response,
    "Map-Reduce": map_reduce_response,
    "Refine": refine_response
}

# Print summaries for comparison
for method, output in summaries.items():
    print("\n")
    print(f"{method} Summary")
    print("\n")
    print(output)

# - Prompt-Based: Uses a custom prompt to summarize text directly.
# - Stuff Chain: Combines all documents into one prompt and summarizes them in a single LLM call.
# - Map-Reduce: Summarizes chunks separately and combines their summaries.
# - Refine: Starts with an initial summary and iteratively updates it using subsequent chunks.


## Task 13: Build a Summarization Function

In [ ]:
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.summarize import load_summarize_chain

def summarize_document(text, method='map_reduce'):
    # Prompt based Summarization
    if method == 'prompt':
        prompt = PromptTemplate(template='''"Write a concise summary of the following text:\n\n{text}"''')
        chain = prompt | llm
        return chain.invoke({"text": text}).content

    # Convert text into a LangChain document
    documents = [Document(page_content=text)]

    # Split documents for long-text methods
    if method in ["map_reduce", "refine"]:
        splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
        documents = splitter.split_documents(documents)

    # Stuff, Map-Reduce or Refine summarization
    if method in ["stuff", "map_reduce", "refine"]:
        chain = load_summarize_chain(llm=llm, chain_type=method)
        result = chain.run(documents)
        return result

    # Handle invalid methods
    raise ValueError(
        "Method must be prompt, stuff, map_reduce, or refine"
    )

In [ ]:
# Test
print("MAP-REDUCE SUMMARY:\n")
print(summarize_document(text, method="map_reduce"))

## Task 14: Observations & Insights

1. Map-Reduce is a good choice for very long documents because it summarizes smaller chunks independently and combines them into a final summary. Refine is useful when maintaining continuity between chunks is important.


2. Prompt-Based and Stuff summarization are generally faster because they require fewer LLM calls. Map-Reduce and Refine require more calls, increasing processing time and cost, but can handle longer documents. Refine processes chunks sequentially, while Map-Reduce can summarize chunks in parallel.


3. Real World Use case


a. Prompt-Based: Summarizing emails, articles, and short text.

b. Stuff Chain: Summarizing short reports and documents.

c. Map-Reduce Chain: Summarizing long research papers, books, and lengthy reports.

d. Refine Chain: Summarizing meeting transcripts, lengthy interviews, and documents where information builds progressively.